<h1>CME538 - Introduction to Data Science</h1>
<h2>Lecture 4.2 - Data Visualization II</h2>
<p>Why this notebook? You will compare plots made from the same numbers. Change the axes, grouping, colour, and labels to see which comparisons become easier or harder. Build one figure in stages until a reader understands the data without a spoken explanation.</p>
<p>Central question: How do plotting choices change the conclusion a reader draws from fixed data?</p>
<h3>Lecture Structure</h3>
<ol><li><a href="#section1">Scale</a></li>
<li><a href="#section2">Conditioning</a></li>
<li><a href="#section3">Perception</a></li>
<li><a href="#section4">Context</a></li></ol>
<p>Follow the numbered visits from the slides. Run setup once, then continue in order. Each visit ends with a takeaway and a return cue.</p>
<p><a href="../slides/week4_lecture2.pptx">Lecture slides</a>. Data: the lecture procedure counts, the local earnings extract, and World Bank country indicators. See <a href="DATA_SOURCES.md">file definitions and limits</a>. The terrain example ships with Matplotlib.</p>

<h2>Setup Notebook</h2><p>Keep both CSV files and <code>utils.py</code> beside this notebook. All examples run offline. NumPy, pandas, Matplotlib, and Seaborn come from the course environment. No additional package installation is needed.</p><p>Plotting settings and repeated layout helpers live in <code>utils.py</code>. The cells show the scale, grouping, colour, and annotation decisions.</p>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cbook as cbook
import seaborn as sns
import utils

%matplotlib inline
utils.configure_notebook()

<a id='section1'></a><h2>1. Scale</h2>
<p>Question: How do two vertical scales change the apparent comparison?</p>
<p>Use the procedure counts printed in the slides. One row represents a year. Two endpoints show change between those years, not the annual path between them. Keep the series colours fixed throughout the comparison.</p>

In [ ]:
procedures = pd.DataFrame({
    'Year': [2006, 2013],
    'Cancer screenings': [2007371, 935573],
    'Abortions': [289750, 327000],
})
procedures

<p>Give each series a separate vertical axis. <code>twinx()</code> creates a second axis at the same position. The two <code>set_ylim()</code> calls determine where the lines appear.</p>

In [ ]:
fig, left = plt.subplots()
left.plot(procedures['Year'], procedures['Cancer screenings'], marker='o', color=utils.BLUE)
left.set(ylabel='Cancer screenings', ylim=(900000, 2100000), xticks=[2006, 2013], xlabel='Year')
right = left.twinx()
right.plot(procedures['Year'], procedures['Abortions'], marker='o', color=utils.RED)
right.set(ylabel='Abortions', ylim=(280000, 340000))
left.set_title('Separate axes: the lines cross on the page')
right.grid(False)
plt.show()

<p>The apparent crossing does not mean the counts became equal. Read the separate scales. Put both series on one axis to compare their magnitudes.</p>

In [ ]:
procedures.plot(x='Year', y=['Cancer screenings', 'Abortions'], marker='o', color=[utils.BLUE, utils.RED])
plt.ylim(0, 2200000)
plt.xticks([2006, 2013])
plt.ylabel('Recorded procedures')
plt.title('Shared axis: cancer screenings exceed abortions in both years')
plt.show()

<h3>Does zero belong on the axis?</h3><p>Bar length represents magnitude, so a truncated baseline distorts the ratio of bar lengths. Compare the same two abortion counts. The reference at zero matters for a bar chart.</p>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax in axes:
    ax.bar(procedures['Year'].astype(str), procedures['Abortions'], color=utils.RED)
    ax.set(xlabel='Year', ylabel='Recorded procedures')
axes[0].set(title='Truncated bars', ylim=(280000, 340000))
axes[1].set(title='Bars from zero', ylim=(0, 340000))
plt.show()

<p>A line plot encodes position rather than length from a baseline. A labelled, restricted range helps inspect a small change. Show the broader comparison first and identify the zoom.</p>

In [ ]:
procedures.plot(x='Year', y='Abortions', marker='o', color=utils.RED, legend=False)
plt.ylim(280000, 340000)
plt.xticks([2006, 2013])
plt.ylabel('Recorded procedures')
plt.title('Zoomed line view: change in abortions between two years')
plt.show()

<h3>Use space to show the observations</h3><p>Read the World Bank extract. Each row represents a country or economy. Select 2016 income per person and the recorded female adult-literacy rate. Literacy observations come from 2005–2014, so this is not a same-year comparison.</p>

In [ ]:
world = pd.read_csv('world_bank.csv', index_col=0)
countries = world[['Country', 'Continent',
                   'Gross national income per capita, Atlas method: $: 2016',
                   'Adult literacy rate: Female: % ages 15 and older: 2005-14']].copy()
countries.columns = ['country', 'continent', 'income', 'female_literacy']
countries.info()

<p>Keep complete pairs and positive income values before using a log axis. The retained sample does not represent every economy.</p>

In [ ]:
countries = countries.dropna(subset=['income', 'female_literacy'])
countries = countries[countries['income'] > 0]
print('Complete positive-income pairs:', len(countries))
countries.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax in axes:
    sns.scatterplot(data=countries, x='income', y='female_literacy', ax=ax, alpha=0.6)
    ax.set(xlabel='GNI per person (2016 US$)', ylabel='Female adult literacy (%)', ylim=(0, 105))
axes[0].set_title('Linear income axis')
axes[1].set_xscale('log')
axes[1].set_title('Log income axis')
plt.show()

<p>The log axis separates countries crowded near the left edge of the linear plot. Equal horizontal distances now represent equal income ratios, not equal dollar differences. The literacy values do not change. These country-level observations do not identify an effect of income on literacy.</p>

<p>Takeaway: Shared units do not guarantee shared scales. Start bars at zero. Label zoomed views. Log spacing reveals low-income differences without changing the country records.</p><p>Back to the slides: Conditioning.</p>

<a id='section2'></a><h2>2. Conditioning</h2>
<p>Question: Which grouping makes the comparison explicit?</p>
<p>Conditioning means comparing observations within a chosen group. The earnings file records one median per education and gender group. These ten medians are summaries, not ten individual salaries. The extract does not record its reference year, so do not treat the values as current earnings.</p>
<p>The previous section changed the coordinate system. This section defines the groups before choosing how to compare their summaries.</p>

In [ ]:
earnings = pd.read_csv('median_weekly_earnings.csv')
earnings

<p>Replace education codes with readable labels. Education has a natural order, so retain this order in every plot.</p>

In [ ]:
education_labels = {1: 'Less than HS', 2: 'High school', 3: 'Some college',
                    4: "Bachelor's", 5: 'Advanced'}
education_order = list(education_labels.values())
earnings['education'] = earnings['educ'].map(education_labels)
earnings['education'] = pd.Categorical(earnings['education'], categories=education_order, ordered=True)
earnings

<h3>Compare within education levels</h3><p>Grouped bars compare the two gender summaries at each education level. Disable uncertainty bars because the file supplies one median per group, not the underlying sample.</p>

In [ ]:
sns.barplot(data=earnings, x='education', y='income', hue='gender',
            order=education_order, palette=utils.GENDER_COLORS, errorbar=None)
plt.title('Compare median earnings within each education level')
plt.xlabel('Education level')
plt.ylabel('Median weekly earnings (US$)')
plt.legend(title='Recorded gender')
plt.show()

<h3>Superposition: share the same axes</h3><p>A pivot places education levels in rows and the two groups in columns. Lines connect ordered categories to support comparison. They do not imply continuous education levels.</p>

In [ ]:
earnings_wide = earnings.pivot(index='education', columns='gender', values='income')
earnings_wide

In [ ]:
earnings_wide[['Men', 'Women']].plot(marker='o', color=[utils.BLUE, utils.RED])
plt.title('Shared axes: compare the gap within each level')
plt.xlabel('Education level')
plt.ylabel('Median weekly earnings (US$)')
plt.ylim(0, 1500)
plt.show()

<a id="optional"></a><h3>Absolute and relative differences</h3><p>Question: Do dollar gaps and percentage gaps rank education levels in the same order?</p><p>Subtract the women's median from the men's median within each education level. Divide by the women's median for a percentage comparison. The denominator names the reference group.</p>

In [ ]:
absolute_gap = earnings_wide['Men'] - earnings_wide['Women']
relative_gap = absolute_gap / earnings_wide['Women'] * 100
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
absolute_gap.plot.bar(ax=axes[0], color=utils.BLUE, rot=25)
axes[0].set(title='Absolute difference', ylabel='Men minus women (US$/week)', xlabel='Education')
relative_gap.plot.bar(ax=axes[1], color=utils.RED, rot=25)
axes[1].set(title='Relative difference', ylabel='Difference / women’s earnings (%)', xlabel='Education')
plt.show()

<p>From some college to a bachelor's degree, the dollar gap rises from $211 to $284 per week. The percentage gap falls from 31.9% to 29.4% because the reference earnings also rise. Choose dollars for a difference in amounts. Choose percentages for a difference relative to a stated base. These comparisons describe group medians, not a change in one person's pay.</p>

<h3>Juxtaposition: separate panels</h3><p>Keep the same vertical limits, category order, and colours. Separate panels reduce overlap, but comparing the groups requires looking across panels.</p>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, gender in zip(axes, ['Men', 'Women']):
    earnings_wide[gender].plot(marker='o', ax=ax, color=utils.GENDER_COLORS[gender])
    ax.set(title=gender, xlabel='Education level', ylabel='Median weekly earnings (US$)', ylim=(0, 1500))
    ax.tick_params(axis='x', rotation=25)
plt.show()

<h3>Order depends on the variable</h3><p>Education follows its ordinal order. Continents have no natural numerical order. Order continent groups by their median recorded income.</p><p>The question now concerns income, so literacy no longer determines which rows belong in the comparison. Use every economy with recorded positive income and a continent label. Each economy receives equal weight. The box plots describe economies in this extract, not the income distribution of individual people.</p>

In [ ]:
income_countries = world[['Continent',
                          'Gross national income per capita, Atlas method: $: 2016']].dropna()
income_countries.columns = ['continent', 'income']
income_countries = income_countries[income_countries['income'] > 0]
print('Economies with positive income:', len(income_countries))

In [ ]:
continent_order = income_countries.groupby('continent')['income'].median().sort_values().index
continent_order

In [ ]:
sns.boxplot(data=income_countries, x='income', y='continent', order=continent_order, color=utils.BLUE)
plt.xscale('log')
plt.title('Order nominal groups by a stated summary')
plt.xlabel('GNI per person (2016 US$, logarithmic scale)')
plt.ylabel('Continent')
plt.show()

<p>Takeaway: Grouping identifies the comparison. Shared axes favour within-level differences. Separate panels favour within-group patterns. Preserve ordinal order and explain any ordering of nominal categories. State whether your comparison uses amounts or relative differences. Select the rows needed for the current question.</p><p>Back to the slides: Perception.</p>

<a id='section3'></a><h2>3. Perception</h2>
<p>Question: Which visual features represent differences accurately?</p>
<p>Start with a smooth numerical gradient. Both strips below show exactly the same values. Only the colour map changes. Compare the apparent bands in jet with the ordered progression in viridis.</p>

In [ ]:
gradient = np.linspace(0, 1, 256).reshape(1, -1)
fig, axes = plt.subplots(2, 1, figsize=(9, 3))
for ax, cmap in zip(axes, ['jet', 'viridis']):
    ax.imshow(gradient, aspect='auto', cmap=cmap, extent=(0, 1, 0, 1))
    ax.set(title=cmap, yticks=[], xlabel='Value')
plt.show()

<h3>Check brightness before reading terrain</h3><p>Question: Does brightness increase as the encoded value increases?</p><p>Relative luminance measures the light contribution of each displayed colour on a scale from black to white. The helper converts stored RGB values to linear light before calculating luminance. Use the result to check ordering, not to prove perceptual uniformity or accessibility.</p>

In [ ]:
utils.plot_colormap_luminance(['jet', 'viridis'])

<p>Viridis increases in luminance as the value increases. Jet becomes brighter and then darker. A bright band in jet therefore need not mark the largest value. Return to the gradient strips, then examine the terrain with this difference in mind.</p>

<h3>Compare colours on terrain data</h3><p>The Jacksboro fault elevation grid ships with Matplotlib. Each cell contains an elevation in metres. Use identical value limits for both colour maps. Terrain contains real edges and slopes. A feature emphasized by colour alone needs inspection of the underlying values.</p>

In [ ]:
with np.load(cbook.get_sample_data('jacksboro_fault_dem.npz', asfileobj=False)) as terrain:
    elevation = terrain['elevation']
print('Grid shape:', elevation.shape)
print('Elevation range (m):', elevation.min(), elevation.max())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, cmap in zip(axes, ['jet', 'viridis']):
    image = ax.imshow(elevation, cmap=cmap, vmin=elevation.min(), vmax=elevation.max())
    ax.set(title=cmap, xticks=[], yticks=[])
    fig.colorbar(image, ax=ax, label='Elevation (m)', shrink=0.8)
plt.show()

<h3>Match colour to meaning</h3><p>Sequential colours represent low to high values. Diverging colours emphasize differences around a reference such as zero. Cyclic colours join at the ends for quantities such as direction. Qualitative colours separate unordered categories.</p>

In [ ]:
utils.show_color_scales()

<h3>Length, angle, and area</h3><p>Question: Estimate the ratio A:B in each panel before reading the values below. The illustration uses the same ratio in every panel. The circle area, not the radius, represents the number.</p>

In [ ]:
utils.compare_encodings(7, 1)

<p>Every panel represents A = 7 and B = 1. Bars provide a common baseline. Pie slices require angle comparisons. Circles require area comparisons. Use position or length when precise comparison matters.</p>

<h3>Read band thickness, not the top edge</h3><p>Use a small numerical illustration to isolate a stacked chart's moving baseline. Central stays at 30 in every year. Compare its band with its separate line. These are illustrative values, not measurements from a city.</p>

In [ ]:
activity = pd.DataFrame({'Year': [2020, 2021, 2022, 2023, 2024],
                         'North': [40, 48, 45, 58, 64],
                         'Central': [30, 30, 30, 30, 30],
                         'South': [25, 27, 30, 32, 35]}).set_index('Year')
activity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
activity.plot.area(ax=axes[0], color=[utils.BLUE, utils.GOLD, utils.GREEN])
axes[0].set(title='Stacked areas: follow Central’s thickness', ylabel='Illustrative units')
activity.plot(ax=axes[1], marker='o', color=[utils.BLUE, utils.GOLD, utils.GREEN])
axes[1].set(title='Lines: Central stays at 30', ylabel='Illustrative units')
for ax in axes:
    ax.set_xticks(activity.index)
plt.show()

<p>The stack makes the total easy to read. Only the bottom series shares a fixed zero baseline. Separate lines make individual trends easier to compare. Ordinary, unstacked histograms do not have this moving-baseline issue.</p>

<p>Takeaway: Choose colours for the variable type. Use position or length for precise comparisons. A stacked chart supports totals, while separate series support component trends.</p><p>Back to the slides: Context.</p>

<a id='section4'></a><h2>4. Context</h2>
<p>Question: What does a reader need to understand a figure without your explanation?</p>
<p>A defensible scale and grouping are insufficient if the reader cannot identify the quantity or population. Return to the earnings comparison. Build the figure in two passes. Keep the numbers and colours fixed. First draw the data without a complete title, units, or source.</p>

In [ ]:
earnings_wide[['Men', 'Women']].plot(marker='o', color=[utils.BLUE, utils.RED])
plt.title('Earnings')
plt.xlabel('')
plt.ylabel('')
plt.show()

<p>The next pass names the quantity, adds units, labels the groups, and points to a numerical difference. The title describes an association. These medians do not measure the effect of earning a degree.</p>

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
earnings_wide[['Men', 'Women']].plot(ax=ax, marker='o', color=[utils.BLUE, utils.RED])
ax.set(title='Higher education groups report higher median weekly earnings',
       xlabel='Education level', ylabel='Median weekly earnings (US$)', ylim=(0, 1650))
ax.legend(title='Recorded gender')
jump = earnings_wide.loc["Bachelor's", 'Men'] - earnings_wide.loc['Some college', 'Men']
ax.annotate(f'Men: ${jump:.0f}/week above some college',
            xy=(3, earnings_wide.loc["Bachelor's", 'Men']), xytext=(1.3, 1510),
            arrowprops={'arrowstyle': '->', 'color': utils.GRAY, 'linewidth': 1.8}, fontsize=11)
fig.text(0.1, 0.01, 'Source: local BLS teaching extract. Reference year absent from the file.\n'
                   'Medians hide within-group variation. Group differences do not establish causation.', fontsize=10)
fig.set_layout_engine('none')
fig.subplots_adjust(bottom=0.22, top=0.87)
plt.show()

<p>A reference line also needs a meaning. The second year's count divided by the first year's count, multiplied by 100, expresses change relative to a base year. Here 100 means no change from 2006.</p>

In [ ]:
indexed = procedures.set_index('Year')
indexed = indexed / indexed.iloc[0] * 100
indexed.plot(marker='o', color=[utils.BLUE, utils.RED])
plt.axhline(100, color=utils.GRAY, linestyle='--', label='2006 baseline')
plt.xticks([2006, 2013])
plt.title('Relative change answers a different question from total counts')
plt.ylabel('Index, 2006 = 100')
plt.legend()
plt.show()

<p>In 2013, the screening index is about 46.6 and the abortion index is about 112.9. Relative to 2006, these correspond to a decrease of about 53.4% and an increase of about 12.9%. The lines no longer compare total procedure counts. They compare change from each series' own base.</p>

<p>Takeaway: A figure needs a clear question, units, group labels, source, and limits. State the meaning of annotations and reference lines. An index compares relative change, not absolute counts.</p><p>Back to the slides: Lecture summary.</p>

<h2>Lecture summary</h2><p>Choose shared scales for magnitude comparisons. Name the groups and select the records needed for the question. State the denominator for a relative difference. Match colour ordering to the variable. Add the source, units, and limits before presenting a conclusion.</p><p>The earnings values stayed fixed while the view changed. Shared axes exposed the gap, percentages changed the comparison, and annotations made the result explicit. Before sharing a plot, ask: What does each mark mean? What comparison does the layout support? What information does the figure omit?</p>